# Analysis for "Anthropometry and biomarkers"

This notebook contains the analysis for, "Anthropometry and biomarkers" (Sejunti et al.). 

## Setup

### Standard modules

In [50]:
import time

import numpy as np
import pandas as pd

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ShuffleSplit
from sklearn.model_selection import cross_validate
from statsmodels.discrete.discrete_model import Logit

### Project-specific functions

In [51]:
def decisiontreeclassification(X, y, classifier, cv=10, random_state=None, rescale=True):
    '''
    This function is used to find the feature importances of the variables by a chosen classifier. Choose either 'decision tree' or 'random forest'. Use rescale=True (the default) to drop age2018 and renormalize the remaining values. Note the default `cv=10`, meaning 10-fold cross validation with deterministic splits. For the manuscript, we use ShuffleSplit instead. 
    '''
    if classifier == 'decision tree':
        clf = DecisionTreeClassifier(
            random_state=random_state, # the default is random seed
            class_weight='balanced' # algorithm considers observations with rare outcome values more important
        )
    elif classifier == 'random forest':
        clf = RandomForestClassifier(
            random_state=random_state,
            class_weight='balanced',
            max_features=3 # default is sqrt, which is $\approx 3.32$. This is easier to say in a methods section.
        )
    else:
        print("Choose one of 'decision tree' or 'random forest'.")
        return None

    output = cross_validate(
        clf, # the classifier
        X, y, # the whole data, in predictor (X) and outcome (y) objects
        cv=cv, # either 10-fold validation (cv=10) or the ShuffleSplit obj. If the latter, will be 100 importances
        return_estimator=True, # return the estimation object so we can extract importances
        n_jobs=-2 # use 1 CPU less than available
    ) 

    # gather the importances from the estimator objects
    all_importances = []
    for estimator in output['estimator']:
        all_importances.append(estimator.feature_importances_)

    # collect and sort the means and std devs (though we don't use the latter)
    mean_importance = np.mean(all_importances, axis=0)
    std_importance = np.std(all_importances, axis=0)
    feature_importances = pd.DataFrame({
        'importance_mean': mean_importance,
        'importance_std': std_importance
    }, index=X.columns).sort_values('importance_mean', ascending=False)

    # return the importances (rescaled if desired, dropping age)
    if rescale:
        imps = feature_importances.drop("age2018")["importance_mean"]
        return imps/sum(imps)
    else:
        return feature_importances

def variable_importance_analysis(X, outcome, classifier, cv=10, random_state=None, rescale=True):
    """
    Return the importances. Convenience wrapper to select an outcome based on a string, then call decisointreeclassification().
    """
    assert outcome in ["diabetes", "hypertension", "ability"], "Set outcome to one of 'diabetes', 'hypertension', or 'ability to work'."
    
    if outcome == "diabetes":
        y = Y['diabHbA1C']
    elif outcome == "hypertension":
        y = Y['htnbin']
    elif outcome == "ability":
        y = Y['binary_ability']
    else:
        print("Set outcome to one of 'diabetes', 'hypertension', or 'ability to work'.")

    imps = decisiontreeclassification(X, y, classifier, cv=cv, random_state=random_state, rescale=rescale)

    return imps

def logisticregressionmodel(X, y): # add printonly arg here to allow storing the model?
    '''
    Fit a logistic regression model to X, predicting y. Print the AIC and model summary (coef values, p-values, etc.), returning None.
    '''
    m = Logit(y, X) # sklearn and statsmodels use different conventions for arg order
    mfit = m.fit()
    print(f"AIC: {np.round(mfit.aic, 2)}")
    print(mfit.summary())
    return None

def logistic_regression_analysis(outcome, importances):
    """
    Following the order in the `importances` argument, show logistic regression model results for five, four, three, two, and one variable models.
    """
    assert outcome in ["diabetes", "hypertension", "ability"], "Set outcome to one of 'diabetes', 'hypertension', or 'ability to work'."

    if outcome == "diabetes":
        y=data['diabHbA1C'] # For Diabetes
    elif outcome == "hypertension":
        y=data['htnbin'] # For Hypertension
    elif outcome == "ability":
        y=data['binary_ability']  # For ability to work
    else:
        print("Set outcome to one of 'diabetes', 'hypertension', or 'ability'.")

    order = list(importances.index)
    
    cols1 = ["age2018"] + order[:5] # top five
    cols2 = ["age2018"] + order[:4] # top four
    cols3 = ["age2018"] + order[:3] # etc.
    cols4 = ["age2018"] + order[:2]
    cols5 = ["age2018"] + order[:1]
    
    X1 = data[cols1]
    X2 = data[cols2]
    X3 = data[cols3]
    X4 = data[cols4]
    X5 = data[cols5]

    print("")
    print("Top five variables in terms of importance (plus control):")
    print(logisticregressionmodel(X1, y))
    print("")
    print("Top four variables in terms of importance (plus control):")
    print(logisticregressionmodel(X2, y))
    print("")
    print("Top three variables in terms of importance (plus control):")
    print(logisticregressionmodel(X3, y))
    print("")
    print("Top two variables in terms of importance (plus control):")
    print(logisticregressionmodel(X4, y))
    print("")
    print("Top variable in terms of importance (plus control):")
    print(logisticregressionmodel(X5, y))

    return None


## Analysis

### Preliminaries

Loading the data, showing summary statistics, and selecting variables for further analysis.

In [52]:
# load merged data set
filename = "15 Dec 25 New data pull narrowed.csv"
data = pd.read_csv(filename)
Xcols = [
    'age2018', 'Weight_kg', 'Height_cm', 'BMIcalc', 'WholeBodyFatPerc', 'TrunkPerc', 
    'waistc', 'hipc', 'waisthp', 'waistht', 'meangripboth', 'tsf' 
]
BPcols = ['BP_Sys_mmHg', 'BP_Dia_mmHg']
Ycols = [
    'binary_ability', 'diabHbA1C', 'htnbin'
]

# Drop individual w/ tsf==0
data.drop(data[data['tsf'] == 0].index, inplace=True)

# multiply ratios by 100 to aid interpretation
data['waisthp'] = data['waisthp']*100
data['waistht'] = data['waistht']*100

print("Predictor variables")
print(
    data[Xcols + BPcols].agg(
        func=["mean", "median", "std", "min", "max", lambda x: x.isna().sum()]
    ).round(3).transpose()
)
print("")
print("The column called '<lambda>' is a count of missing values.")
print("")

print("Outcome variables")
Yagg = data[Ycols].agg([
    lambda x: sum(x.dropna())/len(x.dropna()),
    lambda x: sum(x.isna())/len(x)
]).round(3).transpose()
Yagg.columns = ["proportion", "missingness"]
print(Yagg)
print("")
print(np.round(100*data['ability_to_work_count'].value_counts()/data['ability_to_work_count'].dropna().count(), 1))

# show n
data = data.dropna()
print(data.shape[0])

# select columns for further analysis
X = data[Xcols]
Y = data[Ycols]

Predictor variables
                     mean   median     std      min      max  <lambda>
age2018            50.116   50.000  12.271   20.000   88.000       0.0
Weight_kg          52.241   51.400  10.645   25.500   92.100       0.0
Height_cm         148.840  149.000   5.934  115.800  169.000       0.0
BMIcalc            23.516   23.324   4.252   12.128   39.497       0.0
WholeBodyFatPerc   30.910   31.300   7.640    8.000   52.100       0.0
TrunkPerc          24.829   25.800   9.673    5.000   47.100       0.0
waistc             84.239   84.455  11.214   52.388  124.460       2.0
hipc               91.427   90.805   8.274   68.580  124.460       4.0
waisthp            92.082   92.500   8.756   68.750  123.636       4.0
waistht            56.622   57.020   7.459   36.129   86.431       2.0
meangripboth       15.741   15.550   4.122    5.633   34.967       3.0
tsf                20.409   20.167   7.489    3.000   41.667       1.0
BP_Sys_mmHg       130.960  126.000  23.598   12.000  233.

### Variable importance analysis

In [53]:
# set up splitter for cross-validation
# uses the same X and y size as 10-fold cross validation, but randomly selects rows rather than using a deterministic split. This option is analogous to bootstrapping and in this data tends to produce more stable results.
splitter = ShuffleSplit(n_splits=100, test_size=0.1)

In [54]:
# compute importances
## lists of importances, to check variability
print("Computing diabetes classifiers...")
## for both DecisionTree and RandomForest, repeat the variable importance analysis 5 times with the same parameters, saving the results.
## This code takes some time to run: around 131.81 s (2 min 12 sec) on 4 x Intel Core i3-5010U CPU @ 2.10GHz
t1 = time.time()
diabetes_DT = [
    variable_importance_analysis(X, "diabetes", "decision tree", cv=splitter, random_state=None) for _ in range(5)
]
diabetes_RF = [
    variable_importance_analysis(X, "diabetes", "random forest", cv=splitter, random_state=None) for _ in range(5)
]
## Store one run of each set of importances, for reproducibility, fixing the random seed
imp_diabetes_DT = variable_importance_analysis(X, "diabetes", "decision tree", cv=splitter, random_state=42)
imp_diabetes_RF = variable_importance_analysis(X, "diabetes", "random forest", cv=splitter, random_state=42)
t2 = time.time()
print(f"Run time: {np.round(t2 - t1, 2)}")

Computing diabetes classifiers...
Run time: 197.37


In [55]:
## same for hypertension
print("Computing hypertension classifiers...")
# 2 min 25 sec
t1 = time.time()
hypertension_DT = [
    variable_importance_analysis(X, "hypertension", "decision tree", cv=splitter, random_state=None) for _ in range(5)
]
hypertension_RF = [
    variable_importance_analysis(X, "hypertension", "random forest", cv=splitter, random_state=None) for _ in range(5)
]
imp_hypertension_DT = variable_importance_analysis(X, "hypertension", "decision tree", cv=splitter, random_state=42)
imp_hypertension_RF = variable_importance_analysis(X, "hypertension", "random forest", cv=splitter, random_state=42)
t2 = time.time()
print(f"Run time: {np.round(t2 - t1, 2)}")

Computing hypertension classifiers...
Run time: 202.41


In [56]:
## same for ability to work
print("Computing ability to work classifiers...")
# 2 min 25 sec
t1 = time.time()
ability_DT = [
    variable_importance_analysis(X, "ability", "decision tree", cv=splitter, random_state=None) for _ in range(5)
]
ability_RF = [
    variable_importance_analysis(X, "ability", "random forest", cv=splitter, random_state=None) for _ in range(5)
]
imp_ability_DT = variable_importance_analysis(X, "ability", "decision tree", cv=splitter, random_state=42)
imp_ability_RF = variable_importance_analysis(X, "ability", "random forest", cv=splitter, random_state=42)
t2 = time.time()
print(f"Run time: {np.round(t2 - t1, 2)}")

Computing ability to work classifiers...
Run time: 193.74


In [57]:
# Show the importances
print("Assess variability in importances:")
print("Diabetes")
print("Decision tree")
for x in diabetes_DT:
    print(x)
print("")

print("Random forest")
for x in diabetes_RF:
    print(x)
print("")
print("")

print("Hypertension")
print("Decision tree")
for x in hypertension_DT:
    print(x)
print("")

print("Random forest")
for x in hypertension_RF:
    print(x)
print("")
print("")

print("Ability to work")
print("Decision tree")
for x in ability_DT:
    print(x)
print("")

print("Random forest")
for x in ability_RF:
    print(x)

Assess variability in importances:
Diabetes
Decision tree
tsf                 0.116271
BMIcalc             0.113959
waistc              0.112446
meangripboth        0.109964
waisthp             0.105772
Weight_kg           0.098960
Height_cm           0.085765
hipc                0.077074
WholeBodyFatPerc    0.065110
waistht             0.063695
TrunkPerc           0.050984
Name: importance_mean, dtype: float64
waistc              0.120187
waisthp             0.114877
BMIcalc             0.111441
meangripboth        0.108279
Weight_kg           0.106997
tsf                 0.103784
Height_cm           0.083635
hipc                0.071447
waistht             0.069290
WholeBodyFatPerc    0.064107
TrunkPerc           0.045954
Name: importance_mean, dtype: float64
waistc              0.118008
waisthp             0.112593
Weight_kg           0.110878
meangripboth        0.110004
BMIcalc             0.104572
tsf                 0.103274
Height_cm           0.088182
hipc                0.075

### Logistic regression analysis

In [58]:
# this code prints regression results for each logit model. The coefficient values, 95% CIs, and AICs are available in each output.
print("Diabetes")
print("Decision tree-based importance")
logistic_regression_analysis("diabetes", imp_diabetes_DT)
print("Random forest-based importance")
logistic_regression_analysis("diabetes", imp_diabetes_RF)
print("Hypertension")
print("Decision tree-based importance")
logistic_regression_analysis("hypertension", imp_hypertension_DT)
print("Random forest-based importance")
logistic_regression_analysis("hypertension", imp_hypertension_RF)
print("Ability to work")
print("Decision tree-based importance")
logistic_regression_analysis("ability", imp_ability_DT)
print("Random forest-based importance")
logistic_regression_analysis("ability", imp_ability_RF)

Diabetes
Decision tree-based importance

Top five variables in terms of importance (plus control):
Optimization terminated successfully.
         Current function value: 0.370384
         Iterations 6
AIC: 532.02
                           Logit Regression Results                           
Dep. Variable:              diabHbA1C   No. Observations:                  702
Model:                          Logit   Df Residuals:                      696
Method:                           MLE   Df Model:                            5
Date:                Tue, 17 Feb 2026   Pseudo R-squ.:                 0.03284
Time:                        20:27:00   Log-Likelihood:                -260.01
converged:                       True   LL-Null:                       -268.84
Covariance Type:            nonrobust   LLR p-value:                  0.003407
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------

The effects of multicolinearity are apparent in models of diabetes using random forest importances. Specifically, we observed that the waist-to-height and waist-to-hip ratio variables were often estimated to have similar magnitude coefficients, but opposite signs, and low $p$-values. We re-ran importance analysis and logit models for all three outcome variables dropping the lower importance of the two. In the following analysis we retain waist-to-hip for diabetes, waist-to-height for hypertension, and waist-to-hip for ability to do work.

In [59]:
Xcols_dia = [x for x in Xcols if x != 'waistht'] # retain waisthp
Xcols_hyp = [x for x in Xcols if x != 'waisthp'] # retain waistht
Xcols_abl = [x for x in Xcols if x != 'waistht'] # retain waisthp
X_dia = data[Xcols_dia]
X_hyp = data[Xcols_hyp]
X_abl = data[Xcols_abl]

In [60]:
diabetes_RFm = [
    variable_importance_analysis(X_dia, "diabetes", "random forest", cv=splitter, random_state=None) for _ in range(5)
]
imp_diabetes_RFm = variable_importance_analysis(X_dia, "diabetes", "random forest", cv=splitter, random_state=42)

In [61]:
hypertension_RFm = [
    variable_importance_analysis(X_hyp, "hypertension", "random forest", cv=splitter, random_state=None) for _ in range(5)
]
imp_hypertension_RFm = variable_importance_analysis(X_hyp, "hypertension", "random forest", cv=splitter, random_state=42)

In [62]:
ability_RFm = [
    variable_importance_analysis(X_abl, "ability", "random forest", cv=splitter, random_state=None) for _ in range(5)
]
imp_ability_RFm = variable_importance_analysis(X_abl, "ability", "random forest", cv=splitter, random_state=42)

In [63]:
print("Assess variability in importances:")
print("Diabetes")
print("Random forest")
for x in diabetes_RFm:
    print(x)
print("")
print("")

print("Hypertension")
print("Random forest")
for x in hypertension_RFm:
    print(x)
print("")
print("")

print("Ability to work")
print("Random forest")
for x in ability_RFm:
    print(x)

Assess variability in importances:
Diabetes
Random forest
waisthp             0.114381
BMIcalc             0.111746
waistc              0.107109
meangripboth        0.105306
WholeBodyFatPerc    0.101141
Weight_kg           0.099518
TrunkPerc           0.095684
Height_cm           0.092953
tsf                 0.091444
hipc                0.080718
Name: importance_mean, dtype: float64
waisthp             0.114256
BMIcalc             0.112808
waistc              0.109202
meangripboth        0.104106
WholeBodyFatPerc    0.101531
Weight_kg           0.099126
TrunkPerc           0.095258
Height_cm           0.093216
tsf                 0.090502
hipc                0.079994
Name: importance_mean, dtype: float64
waisthp             0.116353
BMIcalc             0.110935
waistc              0.107131
meangripboth        0.104972
WholeBodyFatPerc    0.101938
Weight_kg           0.099962
TrunkPerc           0.094542
Height_cm           0.093126
tsf                 0.091292
hipc                0.079

In [64]:
print("Diabetes")
print("Random forest-based importance")
logistic_regression_analysis("diabetes", imp_diabetes_RFm)
print("")
print("")

print("Hypertension")
print("Random forest-based importance")
logistic_regression_analysis("hypertension", imp_hypertension_RFm)
print("")
print("")

print("Ability to work")
print("Random forest-based importance")
logistic_regression_analysis("ability", imp_ability_RFm)

Diabetes
Random forest-based importance

Top five variables in terms of importance (plus control):
Optimization terminated successfully.
         Current function value: 0.370375
         Iterations 6
AIC: 532.01
                           Logit Regression Results                           
Dep. Variable:              diabHbA1C   No. Observations:                  702
Model:                          Logit   Df Residuals:                      696
Method:                           MLE   Df Model:                            5
Date:                Tue, 17 Feb 2026   Pseudo R-squ.:                 0.03286
Time:                        20:36:03   Log-Likelihood:                -260.00
converged:                       True   LL-Null:                       -268.84
Covariance Type:            nonrobust   LLR p-value:                  0.003390
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------

For diabetes only, also remove waistc (absolute waist circumference).

In [65]:
Xcols_dia_2 = [x for x in Xcols_dia if x != 'waistc'] # also remove waistc
X_dia_2 = data[Xcols_dia_2]
diabetes_RFm_2 = [
    variable_importance_analysis(X_dia_2, "diabetes", "random forest", cv=splitter, random_state=None) for _ in range(5)
]
imp_diabetes_RFm_2 = variable_importance_analysis(X_dia_2, "diabetes", "random forest", cv=splitter, random_state=42)
print("Assess variability in importances:")
print("Diabetes")
print("Random forest")
for x in diabetes_RFm_2:
    print(x)
print("")
print("")
print("Diabetes")
print("Random forest-based importance")
logistic_regression_analysis("diabetes", imp_diabetes_RFm_2)
print("")
print("")

Assess variability in importances:
Diabetes
Random forest
waisthp             0.133927
BMIcalc             0.128159
meangripboth        0.115112
WholeBodyFatPerc    0.114469
Weight_kg           0.110378
TrunkPerc           0.106974
tsf                 0.101640
Height_cm           0.101176
hipc                0.088164
Name: importance_mean, dtype: float64
waisthp             0.133697
BMIcalc             0.128295
meangripboth        0.115953
WholeBodyFatPerc    0.113446
Weight_kg           0.110120
TrunkPerc           0.105581
tsf                 0.102455
Height_cm           0.101443
hipc                0.089009
Name: importance_mean, dtype: float64
waisthp             0.131894
BMIcalc             0.129007
meangripboth        0.115673
WholeBodyFatPerc    0.112918
Weight_kg           0.110726
TrunkPerc           0.105805
Height_cm           0.102004
tsf                 0.101923
hipc                0.090051
Name: importance_mean, dtype: float64
waisthp             0.134442
BMIcalc         